[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Files, Paths and Formats](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)

# Writing Safely &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.


In [1]:

import os
import tempfile
import json
from pathlib import Path
import shutil

scratch = Path("scratch")
scratch.mkdir(exist_ok=True)
notes = scratch / "notes.txt"


def fresh_notes():
    notes.write_text("first\nsecond\nthird\n", encoding="utf-8")
    return notes


**1.** Write `scratch/notes.txt` with three lines, then overwrite it naively and raise
partway through. Print what is left.


In [2]:

fresh_notes()
print("before:", repr(notes.read_text(encoding="utf-8")))

try:
    with open(notes, "w", encoding="utf-8") as f:
        f.write("replacement line one\n")
        raise RuntimeError("interrupted")
except RuntimeError as e:
    print("failed:", e)

print("after: ", repr(notes.read_text(encoding="utf-8")))


before: 'first\nsecond\nthird\n'
failed: interrupted
after:  'replacement line one\n'


One line where there were three, and the two that were replaced are unrecoverable.


**2.** Do the same with a temporary file and `os.replace`, and show the original survived.


In [3]:

fresh_notes()
print("before:", repr(notes.read_text(encoding="utf-8")))

temporary = None
try:
    handle, temporary = tempfile.mkstemp(dir=notes.parent, suffix=".tmp")
    with os.fdopen(handle, "w", encoding="utf-8") as f:
        f.write("replacement line one\n")
        raise RuntimeError("interrupted")
    os.replace(temporary, notes)
except RuntimeError as e:
    print("failed:", e)
finally:
    if temporary is not None:
        Path(temporary).unlink(missing_ok=True)

print("after: ", repr(notes.read_text(encoding="utf-8")))


before: 'first\nsecond\nthird\n'
failed: interrupted
after:  'first\nsecond\nthird\n'


The failure is identical and the file is untouched, because `os.replace` was never reached. The
`finally` removed the partial temporary file.


**3.** Let the safe version complete, and confirm the new content and that no temporary file
remains.


In [4]:

fresh_notes()

handle, temporary = tempfile.mkstemp(dir=notes.parent, suffix=".tmp")
with os.fdopen(handle, "w", encoding="utf-8") as f:
    f.write("alpha\nbeta\n")

os.replace(temporary, notes)

print("after:", repr(notes.read_text(encoding="utf-8")))
print("temporary file gone:", not Path(temporary).exists())
print("nothing left over:  ", sorted(p.suffix for p in scratch.iterdir()))


after: 'alpha\nbeta\n'
temporary file gone: True
nothing left over:   ['.txt']


`os.replace` moved the temporary file rather than copying it, so there is nothing to clean up
when the write succeeds.


**4.** Write a function `safe_write(path, text)` that does the temporary-file dance, cleaning
up in a `finally`.


In [5]:

def safe_write(path, text):
    handle, temporary = tempfile.mkstemp(dir=path.parent, prefix=path.name, suffix=".tmp")
    try:
        with os.fdopen(handle, "w", encoding="utf-8") as f:
            f.write(text)
        os.replace(temporary, path)
    finally:
        Path(temporary).unlink(missing_ok=True)


safe_write(notes, "written by the function\n")

print(repr(notes.read_text(encoding="utf-8")))


'written by the function\n'


`missing_ok=True` is what lets the `finally` run on both paths. On success the file has already
been moved and there is nothing to delete; on failure it is still there and gets removed.


**5.** Extend it to refuse content that is not valid JSON, and show it refusing.


In [6]:

def safe_write_json(path, text):
    handle, temporary = tempfile.mkstemp(dir=path.parent, prefix=path.name, suffix=".tmp")
    try:
        with os.fdopen(handle, "w", encoding="utf-8") as f:
            f.write(text)
        json.loads(Path(temporary).read_text(encoding="utf-8"))   # the check
        os.replace(temporary, path)
        return True
    except json.JSONDecodeError as e:
        print("  refused:", e.msg)
        return False
    finally:
        Path(temporary).unlink(missing_ok=True)


config = scratch / "config.json"
config.write_text('{"version": 1}', encoding="utf-8")

print("valid: ", safe_write_json(config, '{"version": 2}'))
print("now:   ", config.read_text(encoding="utf-8"))
print("broken:", safe_write_json(config, '{"version": '))
print("still: ", config.read_text(encoding="utf-8"))


valid:  True
now:    {"version": 2}
  refused: Expecting value
broken: False
still:  {"version": 2}


The check reads the **temporary file back** rather than checking the string it was given. That
matters: it catches an encoding problem or a truncated write as well as bad content.


**6.** Use `TemporaryDirectory` to create two files, print them, and show the folder is gone
afterward.


In [7]:

with tempfile.TemporaryDirectory() as folder:
    workspace = Path(folder)
    (workspace / "one.txt").write_text("first", encoding="utf-8")
    (workspace / "two.txt").write_text("second", encoding="utf-8")
    print("inside:", sorted(p.name for p in workspace.iterdir()))

print("after: exists =", workspace.exists())


inside: ['one.txt', 'two.txt']
after: exists = False


The folder and everything in it went when the block ended, including if the block had raised.
Nothing is left to clean up on a later run.


In [8]:

shutil.rmtree(scratch)

print("cleaned up:", not scratch.exists())


cleaned up: True


---

&#8592; **Back to:** [Writing Safely](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/09-writing-safely.ipynb)  &nbsp;&middot;&nbsp;  [Files, Paths and Formats Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)
